# Model 3 — ConvoKit + Stylometric Features + Logistic Regression

**Recovered notebook (Nov 10, 2026).** Reconstructed from the code and written results previously shared in ChatGPT. This is a **backup reconstruction**, not the original Colab revision; it does **not** include previously executed cell outputs or temporary runtime caches.

**Goal:** Predict human-written (0) vs AI-generated (1) text using linguistic features. Use the same 8,000/2,000 stratified validation split as the team. Previously reported: Train AUROC **0.8427**, Val AUROC **0.8267**, Kaggle Public AUROC **0.71956**. Those are historical results, **not** new results from running this copy.

Place `train.csv`, `test.csv`, `unlabeled.csv` and `sample_submission.csv` in the Colab current working directory. Do not commit those CSV files to public GitHub.

## 1. Setup
Install ConvoKit, spaCy, and standard ML packages. In Colab, restart the runtime after installing if import errors occur.

In [ ]:
%pip -q install convokit spacy nltk scikit-learn tqdm matplotlib
!python -m spacy download en_core_web_sm

### Load libraries
ConvoKit parses linguistic structure; scikit-learn handles train/validation splitting and classification.

In [ ]:
import os
import re
import string
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import spacy
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
from convokit.text_processing.textParser import process_text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nlp = spacy.load("en_core_web_sm")
RANDOM_STATE = 42
Path("results").mkdir(exist_ok=True)
print("Environment ready; spaCy", spacy.__version__)

## 2. Data inspection
Read the official Kaggle CSV files; validate the label balance and column names.

In [ ]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")
unlabeled_df = pd.read_csv("unlabeled.csv")
sample_submission = pd.read_csv("sample_submission.csv")
print('Train:', train_df.shape, 'Test:', test_df.shape, 'Unlabeled:', unlabeled_df.shape)
print('Columns:', train_df.columns.tolist())
print('Labels:', train_df['label'].value_counts().to_dict())
print('Missing:', train_df.isna().sum().to_dict())
display(train_df.head(3))

## 3. Fixed stratified validation split
Reserve 2,000 examples for validation and train on the other 8,000. Save IDs so teammates can check they are using identical rows.

In [ ]:
train_df = train_df.sort_values("id").reset_index(drop=True)
train_data, val_data = train_test_split(
    train_df, test_size=2000,
    stratify=train_df["label"], random_state=RANDOM_STATE
)
train_data = train_data.sort_values("id").reset_index(drop=True)
val_data = val_data.sort_values("id").reset_index(drop=True)
print('Train:', len(train_data), train_data['label'].value_counts().to_dict())
print('Validation:', len(val_data), val_data['label'].value_counts().to_dict())
train_data[['id']].to_csv('results/train_ids.csv', index=False)
val_data[['id']].to_csv('results/val_ids.csv', index=False)

## 4. Try ConvoKit TextParser
Check its token, part-of-speech and dependency output on one document before large-scale processing.

In [ ]:
sample_text = train_data.iloc[0]['text']
parsed = process_text(sample_text, mode="parse", sent_tokenizer=nltk.sent_tokenize, spacy_nlp=nlp)
print('Sentences:', len(parsed))
display(pd.DataFrame([{'Token':t['tok'], 'POS':t.get('tag'), 'Dependency':t.get('dep')} for t in parsed[0]['toks']]).head(15))

## 5. Define 15 stylometric features
Convert each passage into numerical statistics: sentence length, vocabulary diversity, repetition, POS proportions, dependency distance, punctuation and case. These are hypotheses about useful features, not assumed authorship rules.

In [ ]:
def extract_stylometric_features(text):
    text = str(text)
    parsed = process_text(text, mode="parse", sent_tokenizer=nltk.sent_tokenize, spacy_nlp=nlp)
    sentences = [sentence["toks"] for sentence in parsed if sentence.get("toks")]
    all_tokens = [token for sent in sentences for token in sent]
    word_tokens = [t for t in all_tokens if re.search(r"[A-Za-z]", t["tok"])]
    words = [t["tok"].lower() for t in word_tokens]
    tags = [t.get("tag", "") for t in word_tokens]
    n_words, n_chars = len(words), len(text)
    sentence_lengths = [sum(bool(re.search(r"[A-Za-z]", t["tok"])) for t in s) for s in sentences]
    avg_sentence_length = float(np.mean(sentence_lengths)) if sentence_lengths else 0
    sentence_length_std = float(np.std(sentence_lengths)) if sentence_lengths else 0
    type_token_ratio = len(set(words))/n_words if n_words else 0
    avg_word_length = float(np.mean([len(w) for w in words])) if words else 0
    bigrams = list(zip(words[:-1], words[1:]))
    bigram_counts = Counter(bigrams)
    repeated_bigram_rate = sum(count - 1 for count in bigram_counts.values()) / len(bigrams) if bigrams else 0
    noun_ratio = sum(t.startswith("NN") for t in tags) / max(n_words,1)
    verb_ratio = sum(t.startswith("VB") for t in tags) / max(n_words,1)
    adj_ratio = sum(t.startswith("JJ") for t in tags) / max(n_words,1)
    adv_ratio = sum(t.startswith("RB") for t in tags) / max(n_words,1)
    pronoun_ratio = sum(t in {"PRP","PRP$","WP","WP$"} for t in tags)/max(n_words,1)
    dependency_distances = []
    for sentence in sentences:
        for i, token in enumerate(sentence):
            parent = token.get("up")
            if isinstance(parent, int):
                dependency_distances.append(abs(i-parent))
    avg_dependency_distance = float(np.mean(dependency_distances)) if dependency_distances else 0
    punctuation_ratio = sum(ch in string.punctuation for ch in text) / max(n_chars,1)
    uppercase_ratio = sum(ch.isupper() for ch in text) / max(sum(ch.isalpha() for ch in text),1)
    digit_ratio = sum(ch.isdigit() for ch in text) / max(n_chars,1)
    return {
        "word_count": n_words,
        "avg_sentence_length": avg_sentence_length,
        "sentence_length_std": sentence_length_std,
        "avg_word_length": avg_word_length,
        "type_token_ratio": type_token_ratio,
        "repeated_bigram_rate": repeated_bigram_rate,
        "noun_ratio": noun_ratio,
        "verb_ratio": verb_ratio,
        "adj_ratio": adj_ratio,
        "adv_ratio": adv_ratio,
        "pronoun_ratio": pronoun_ratio,
        "avg_dependency_distance": avg_dependency_distance,
        "punctuation_ratio": punctuation_ratio,
        "uppercase_ratio": uppercase_ratio,
        "digit_ratio": digit_ratio,
    }

### Sanity check
Run on five passages before parsing the whole dataset.

In [ ]:
sample_features_df = pd.DataFrame([extract_stylometric_features(text) for text in train_data['text'].head(5)])
print('Features:', sample_features_df.shape[1])
display(sample_features_df.round(3))

## 6. Extract and cache features
Parse 8,000 training and 2,000 validation documents. Results are checkpointed every 200 passages so a rerun can resume. If you change the feature definition, delete the cached CSV files first.

In [ ]:
def build_feature_matrix(df, output_path):
    output_path = Path(output_path)
    if output_path.exists():
        existing = pd.read_csv(output_path)
        if existing['id'].duplicated().any():
            raise ValueError('Duplicate IDs in cache: '+str(output_path))
        completed_ids = set(existing['id'])
    else:
        completed_ids = set()
    remaining = df[~df['id'].isin(completed_ids)]
    print('Remaining passages:', len(remaining))
    buffer = []
    for row in tqdm(remaining.itertuples(index=False), total=len(remaining)):
        features = extract_stylometric_features(row.text)
        features['id'] = row.id
        buffer.append(features)
        if len(buffer) >= 200:
            pd.DataFrame(buffer).to_csv(output_path, mode='a', header=not output_path.exists(), index=False)
            buffer = []
    if buffer:
        pd.DataFrame(buffer).to_csv(output_path, mode='a', header=not output_path.exists(), index=False)
    result = pd.read_csv(output_path)
    result = df[['id']].merge(result, on='id', how='left', validate='one_to_one')
    assert result.notna().all().all(), 'Missing extracted feature rows'
    return result

train_features = build_feature_matrix(train_data, 'results/train_stylometric_features.csv')
val_features = build_feature_matrix(val_data, 'results/val_stylometric_features.csv')
print('Train features:',train_features.shape,'Validation features:',val_features.shape)

## 7. Train Logistic Regression
Scale the features and fit the classifier using training data only. Fit time here excludes linguistic parsing.

In [ ]:
feature_cols = [col for col in train_features.columns if col != 'id']
X_train, X_val = train_features[feature_cols], val_features[feature_cols]
y_train, y_val = train_data['label'], val_data['label']
assert X_train.notna().all().all() and X_val.notna().all().all()
model = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000, random_state=RANDOM_STATE))
t0=time.perf_counter()
model.fit(X_train, y_train)
fit_seconds=time.perf_counter()-t0
print('Model trained. Features:',len(feature_cols),'Fit seconds:',round(fit_seconds,3))

## 8. Evaluate AUROC
Report training and validation AUROC and the generalization gap. Validation score is based only on the held-out 2,000 labeled examples.

In [ ]:
train_probs = model.predict_proba(X_train)[:,1]
val_probs = model.predict_proba(X_val)[:,1]
train_auc = roc_auc_score(y_train,train_probs)
val_auc = roc_auc_score(y_val,val_probs)
print(f'Train AUROC: {train_auc:.4f}')
print(f'Val AUROC:   {val_auc:.4f}')
print(f'Gap:         {train_auc-val_auc:.4f}')

## 9. Save validation predictions
Save the ID-aligned held-out probabilities for team comparison or ensemble experiments. Keep row-level validation predictions out of the public course GitHub repository.

In [ ]:
val_predictions=val_data[['id','label']].copy()
val_predictions['p_ai']=val_probs
val_predictions.to_csv('results/model3_val_predictions.csv',index=False)
model_summary=pd.DataFrame([{'model':'ConvoKit Stylometry + LR','train_auc':train_auc,'val_auc':val_auc,'train_fit_seconds':fit_seconds,'n_features':len(feature_cols)}])
model_summary.to_csv('results/model3_summary.csv',index=False)
display(model_summary.round(4))

## 10. Train final model on all 10,000 labeled examples
Only after choosing the final configuration, combine the existing training/validation feature matrices and retrain the full pipeline. This does not replace the original held-out validation result.

In [ ]:
all_features=pd.concat([train_features,val_features], ignore_index=True)
assert all_features['id'].is_unique
assert set(all_features['id'])==set(train_df['id'])
full_features=train_df[['id']].merge(all_features,on='id',how='left',validate='one_to_one')
X_full=full_features[feature_cols]
y_full=train_df['label']
assert X_full.notna().all().all()
final_model=clone(model)
final_model.fit(X_full,y_full)
print('Final model trained on',len(X_full),'rows')

## 11. Extract features and generate a Kaggle submission
Use the exact same features for test examples, then write `id,label` where label is P(AI), matching the official sample submission. This does **not** upload to Kaggle automatically.

In [ ]:
test_features=build_feature_matrix(test_df,'results/test_stylometric_features.csv')
X_test=test_features[feature_cols]
test_probs=final_model.predict_proba(X_test)[:,1]
predictions=pd.DataFrame({'id':test_df['id'],'label':test_probs})
submission=sample_submission[['id']].merge(predictions,on='id',how='left',validate='one_to_one')
assert len(submission)==len(test_df)==3000
assert submission.columns.tolist()==['id','label']
assert submission['label'].notna().all() and submission['label'].between(0,1).all()
submission.to_csv('submission_model3.csv',index=False)
print('Created submission_model3.csv')
display(submission.head())

### Download important outputs
Download the notebook itself using **File → Download → Download .ipynb**. Optionally download `submission_model3.csv` and `results/model3_val_predictions.csv`. Runtime files can disappear if the Colab session resets.

In [ ]:
from google.colab import files
files.download('submission_model3.csv')